# CS Brand Report — Česká spořitelna

Generuje `report_rating_2026_cs.html` — přehled ratingů poboček v novém vizuálním designu.

**Předpoklady:**
- Existuje `report_rating_2026_staticky.pkl` (vygenerovaný z `internirating_report.py`)
- Soubory `internirating_report.py` a `internirating_report_cs.py` jsou ve stejném adresáři jako tento notebook

In [ ]:
# ── 0. Konfigurace ──────────────────────────────────────────────────────────
PKL_PATH     = "report_rating_2026_staticky.pkl"   # vstupní data
OUTPUT_PREFIX = "report_rating_2026"               # výstupní soubor bude <prefix>_cs.html
CS_ACCENT    = "#245375"                           # dominantní barva (Stone); změň dle potřeby
#  Ostatní volby:
#  Bright Blue #2870ED | Teal #02A3A4 | Forest #028661 | Apple #0CB43F
#  Orange #FF6130      | Pink #EB4C79 | Aubergine #721C7A | Stone #245375

In [ ]:
# ── 1. Importy ───────────────────────────────────────────────────────────────
import os, sys, io
import importlib.util as _ilu
import pandas as pd

# Přidej adresář notebooku do sys.path
_NB_DIR = os.getcwd()
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

In [ ]:
# ── 2. Načti data z PKL ──────────────────────────────────────────────────────
if not os.path.exists(PKL_PATH):
    raise FileNotFoundError(
        f"PKL soubor nenalezen: {PKL_PATH}\n"
        "Nejprve spusť internirating_report.py nebo hlavní notebook."
    )

df_sorted = pd.read_pickle(PKL_PATH)
print(f"✅ Načteno {len(df_sorted)} poboček z {PKL_PATH}")
df_sorted.head(3)

In [ ]:
# ── 3. Načti funkce z internirating_report.py ────────────────────────────────
#
# internirating_report.py je skript (ne modul), spouští kód na úrovni modulu.
# Načteme ho přes importlib; chyby způsobené chybějícími datovými soubory
# potlačíme — funkce jsou definovány před spouštěcím kódem, takže budou dostupné.

_ir_path = os.path.join(_NB_DIR, "internirating_report.py")
if not os.path.exists(_ir_path):
    raise FileNotFoundError(f"Nenalezen: {_ir_path}")

_spec = _ilu.spec_from_file_location("_ir_mod", _ir_path)
_mod  = _ilu.module_from_spec(_spec)

# Spusť modul se suprimovaným výstupem; chyby ze spouštěcího kódu ignoruj
_captured = io.StringIO()
_old_out, _old_err = sys.stdout, sys.stderr
sys.stdout = sys.stderr = _captured
try:
    _spec.loader.exec_module(_mod)
except Exception:
    pass
finally:
    sys.stdout, sys.stderr = _old_out, _old_err

# Vyber potřebné funkce a proměnné
generate_filterable_table         = _mod.generate_filterable_table
generate_specialiste_summary_table = _mod.generate_specialiste_summary_table
DBS_DATE                          = getattr(_mod, 'DBS_DATE', "")
df_specialiste_detail             = getattr(_mod, 'df_specialiste_detail', None)

print(f"✅ Funkce načteny  |  DBS_DATE: {DBS_DATE}")
if df_specialiste_detail is not None and not df_specialiste_detail.empty:
    print(f"   df_specialiste_detail: {len(df_specialiste_detail)} řádků")
else:
    print("   df_specialiste_detail: nedostupné (sekce obsazení pozic bude prázdná)")

In [ ]:
# ── 4. Vygeneruj CS brand report ─────────────────────────────────────────────
from importlib import reload as _reload
import internirating_report_cs as _cs
_reload(_cs)  # vždy načti aktuální verzi souboru

out_file = _cs.generate_cs_brand_report(
    df_sorted              = df_sorted,
    dbs_date               = DBS_DATE,
    output_prefix          = OUTPUT_PREFIX,
    fn_filterable_table    = generate_filterable_table,
    fn_specialiste_summary = generate_specialiste_summary_table,
    df_specialiste_detail  = df_specialiste_detail,
    accent                 = CS_ACCENT,
)

In [ ]:
# ── 5. Zobraz odkaz na výsledný soubor ──────────────────────────────────────
from IPython.display import display, HTML

abs_path = os.path.abspath(out_file)
display(HTML(
    f'<p style="font-family:sans-serif;font-size:1rem;">'
    f'✅ Report vygenerován: '
    f'<a href="{out_file}" target="_blank" style="color:#245375;font-weight:700;">'
    f'{out_file}</a></p>'
    f'<p style="font-size:0.8rem;color:#666;">{abs_path}</p>'
))